# 04. Reranking
---

## Información

### Objetivo

Aplicar un modelo CrossEncoder para reordenar los fragmentos recuperados durante la etapa de Retrieval, priorizando aquellos más relevantes para responder la consulta del usuario.

### Objetivos específicos

* Cargar el modelo CrossEncoder.
* Aplicar reranking sobre los fragmentos recuperados.
* Ordenar los resultados según su relevancia.
* Seleccionar el contexto que será enviado al modelo de lenguaje.
* Validar que el reranking mejora la calidad del contexto recuperado.

### Entradas

* Resultados obtenidos durante la etapa de Retrieval.
* Modelo `cross-encoder/ms-marco-MiniLM-L-6-v2`.

### Salidas

* Fragmentos reordenados por relevancia.
* Contexto optimizado para la etapa de generación.

---
## Etapa 1. Preparación de modelos y datos

### Preparación de entorno

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

vector_store_path = project_root / "data" / "vector_store"

### Importación de módulos de Retrieval
Se importan las funciones necesarias para conectarse al vector store y realizar la recuperación semántica de fragmentos.

In [ ]:
from src.processing.vector_store import (
    get_or_create_collection,
)

from src.processing.retriever import retrieve_context

### Carga de la colección vectorial
Se carga la colección persistente de ChromaDB creada durante la etapa de embeddings.

Esta colección contiene los embeddings generados a partir de los chunks del documento oficial.

In [ ]:
collection = get_or_create_collection(
    collection_name="alessia_collection",
    vector_store_path=vector_store_path,
)

print(f"Colección: {collection.name}")
print(f"Cantidad de vectores: {collection.count()}")

### Carga del modelo de embeddings
Se carga el mismo modelo utilizado durante la etapa de indexación para transformar la consulta del usuario en un vector semántico.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

### Definición de consulta de prueba
Se define una consulta representativa que será utilizada para evaluar el proceso de Retrieval y posteriormente comparar el resultado después del reranking.

In [ ]:
query = "¿Qué medidas debe tomar la comunidad ante una inundación?"

### Recuperación inicial mediante Retrieval
Se recuperan los fragmentos más similares a la consulta utilizando la búsqueda semántica sobre ChromaDB.

Estos resultados serán utilizados como entrada para el modelo CrossEncoder.

In [ ]:
retrieved_chunks = retrieve_context(
    query=query,
    collection=collection,
    model=embedding_model,
    k=5,
)

### Carga del modelo CrossEncoder
Se carga el modelo encargado de evaluar la relación entre la consulta y cada fragmento recuperado.

A diferencia de los embeddings, el CrossEncoder analiza ambos textos conjuntamente para obtener un puntaje de relevancia.

In [ ]:
from sentence_transformers import CrossEncoder

from src.processing.reranker import rerank_chunks


reranker_model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

---
## Etapa 2. Comparación Retrieval vs Reranking

En esta etapa se comparan los resultados obtenidos mediante búsqueda semántica con los resultados después de aplicar el modelo CrossEncoder.

El objetivo es observar cómo el reranking modifica el orden de relevancia de los fragmentos recuperados antes de enviarlos al modelo de lenguaje.

### Resultados obtenidos mediante Retrieval

Los siguientes fragmentos corresponden a los primeros resultados recuperados utilizando similitud semántica entre la consulta y los embeddings almacenados.

In [ ]:
for i, chunk in enumerate(
    retrieved_chunks,
    start=1,
):
    print("=" * 80)
    print(f"Ranking Retrieval: {i}")
    print()
    print(chunk.content[:500])

### Aplicación del modelo CrossEncoder

Los fragmentos recuperados son evaluados nuevamente por el modelo CrossEncoder.

Este modelo calcula un puntaje de relevancia considerando simultáneamente la consulta del usuario y cada fragmento recuperado.

In [ ]:
reranked_results = rerank_chunks(
    query=query,
    chunks=retrieved_chunks,
    model=reranker_model,
    top_k=3,
)

### Resultados después del Reranking

Los fragmentos son mostrados nuevamente ordenados según el puntaje obtenido por el modelo CrossEncoder.

In [ ]:
for i, chunk in enumerate(
    reranked_results,
    start=1,
):
    print("=" * 80)
    print(f"Ranking Reranking: {i}")
    print(f"Score: {chunk.metadata['rerank_score']:.4f}")
    print()
    print(chunk.content[:500])

### Comparación de rankings

La siguiente comparación permite observar si el modelo CrossEncoder modificó el orden de relevancia respecto a la recuperación inicial.

In [ ]:
print("Consulta:")
print(query)

print("\nRanking final seleccionado:")

for i, chunk in enumerate(
    reranked_results,
    start=1,
):
    print(
        f"{i}. Score: {chunk.metadata['rerank_score']:.4f}"
    )

## ✅ Conclusiones

En esta etapa aprendimos que:

- Retrieval permite obtener candidatos relevantes mediante similitud semántica.
- El modelo CrossEncoder realiza una segunda evaluación considerando la relación directa entre la consulta y cada fragmento.
- El reranking permite priorizar información más adecuada antes de enviarla al modelo de lenguaje.
- Esta etapa reduce ruido en el contexto y mejora la calidad potencial de las respuestas generadas por Alessia.